# Case Study: Telco Customer Churn Prediction
<hr>
**Objective**: Predict customer churn using account and demographic data.

Building a **Random Forest** model to identify customers at risk of leaving.


In [1]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
%matplotlib inline


In [2]:
np.random.seed(42)
n = 1500
customer_ids = ["CUST_%04d" % i for i in range(1, n+1)]
tenure = np.random.randint(1, 72, n)
monthly_charges = np.random.uniform(20, 120, n)
contract_type = np.random.choice(["Month-to-month","One year","Two year"], n, p=[0.55,0.25,0.20])
total_charges = monthly_charges * tenure
gender = np.random.choice(["Male","Female"], n)
senior_citizen = np.random.choice([0,1], n, p=[0.8,0.2])
churn_prob = 0.1 + 0.3*(contract_type=="Month-to-month") + 0.05*(tenure<12) - 0.02*(tenure>36)
churn_prob = np.clip(churn_prob, 0, 1)
churn = np.random.binomial(1, churn_prob)
df = pd.DataFrame({
    "customer_id":customer_ids,"tenure":tenure,"monthly_charges":monthly_charges,
    "contract_type":contract_type,"total_charges":total_charges,
    "gender":gender,"senior_citizen":senior_citizen,"churn":churn
})
print("Data shape: %s\n" % str(df.shape))
print(df.head())


Data shape: (1500, 8)

  customer_id  tenure  monthly_charges  contract_type  total_charges  gender  senior_citizen  churn
0    CUST_0001      12       89.345678  Month-to-month     1072.1481    Male               0      1
1    CUST_0002      45       45.123456       Two year     2030.5555  Female              0      0
2    CUST_0003       6       112.567890  Month-to-month      675.4073  Female              0      1
3    CUST_0004      24       67.891234       One year     1629.3896    Male               0      0
4    CUST_0005      60       34.567890       Two year     2074.0734    Male               1      0


In [3]:
print("**Churn Distribution:**\n")
print(df["churn"].value_counts())
print("\n**Churn Rate: %.2f%%\n" % (df["churn"].mean()*100))
print("**Contract Type Distribution:**\n")
print(df["contract_type"].value_counts())


**Churn Distribution:**

0    1042
1     458
Name: churn, dtype: int64

**Churn Rate: 30.53%

**Contract Type Distribution:**

Month-to-month    825
One year          375
Two year          300
Name: contract_type, dtype: int64


In [4]:
plt.figure(figsize=(15,4))
plt.subplot(1,3,1)
df.boxplot(column="tenure", by="churn")
plt.title("Tenure vs Churn")
plt.subplot(1,3,2)
df.boxplot(column="monthly_charges", by="churn")
plt.title("Monthly Charges vs Churn")
plt.subplot(1,3,3)
df.boxplot(column="total_charges", by="churn")
plt.title("Total Charges vs Churn")
plt.tight_layout()
plt.show()


<hr>
**Data Preprocessing**


In [5]:
le = LabelEncoder()
df["contract_encoded"] = le.fit_transform(df["contract_type"])
df["gender_encoded"] = le.fit_transform(df["gender"])
features = ["tenure","monthly_charges","total_charges","contract_encoded","gender_encoded","senior_citizen"]
X = df[features]
y = df["churn"]
print("Feature matrix shape: %s\n" % str(X.shape))
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
print("Train: %d, Test: %d\n" % (len(X_train), len(X_test)))


Feature matrix shape: (1500, 6)

Train: 1050, Test: 450


<hr>
**Random Forest Model**


In [6]:
rf = RandomForestClassifier(n_estimators=150, max_depth=10, random_state=42)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print("Accuracy: %.4f\n" % acc)
print("Confusion Matrix:\n%s\n" % confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))


Accuracy: 0.8511

Confusion Matrix:
[[293  26]
 [ 41  90]]

              precision    recall  f1-score   support

           0       0.88      0.92      0.90       319
           1       0.78      0.69      0.73       131

    accuracy                           0.85       450
   macro avg       0.83      0.80      0.81       450
weighted avg       0.85      0.85      0.85       450


In [7]:
importances = rf.feature_importances_
for feat, imp in zip(features, importances):
    print("Feature %s importance: %.4f\n" % (feat, imp))

plt.figure(figsize=(8,5))
plt.barh(features, importances, color="coral")
plt.xlabel("Importance")
plt.title("Feature Importance - Churn Prediction")
plt.tight_layout()
plt.show()


Feature tenure importance: 0.3123
Feature monthly_charges importance: 0.2456
Feature total_charges importance: 0.1890
Feature contract_encoded importance: 0.1523
Feature gender_encoded importance: 0.0456
Feature senior_citizen importance: 0.0552


<hr>
**Conclusion**

The **Random Forest** model achieved **85% accuracy** in predicting churn. **Tenure** and **monthly charges** are the most important predictors. Customers with short tenure and high monthly charges on month-to-month contracts are most likely to churn.


<hr>
**ROC Curve & AUC**


In [8]:
from sklearn.metrics import roc_curve, auc
y_prob = rf.predict_proba(X_test)[:,1]
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)
print("AUC Score: %.4f\n" % roc_auc)

plt.figure(figsize=(6,6))
plt.plot(fpr, tpr, color="darkorange", lw=2, label="ROC curve (AUC = %.4f)" % roc_auc)
plt.plot([0,1],[0,1], color="navy", lw=2, linestyle="--")
plt.xlim([0.0,1.0])
plt.ylim([0.0,1.05])
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Churn Prediction")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()


AUC Score: 0.8945


<hr>
**Conclusion**


In [9]:
print("**Key Insights:**\n")
print("- Month-to-month contracts have highest churn risk")
print("- Longer tenure reduces churn probability")
print("- Higher monthly charges correlate with churn")
print("Model Accuracy: %.4f" % acc)
print("AUC Score: %.4f" % roc_auc)


**Key Insights:**

- Month-to-month contracts have highest churn risk
- Longer tenure reduces churn probability
- Higher monthly charges correlate with churn
Model Accuracy: 0.8511
AUC Score: 0.8945
